# Assignment 6.1 – Decision Trees, Random Forests & Extra Trees
**Dataset:** UCI Car Evaluation (`car.data`)  
Covers **Activity 4.01** (Decision Tree) and **Activity 4.02** (Random Forest & Extremely Randomized Trees).

*Dataset citation: Dua, D., & Graff, C. (2017). UCI Machine Learning Repository.*

## Activity 4.01 – Car Data Classification
### Step 1: Import libraries and load the dataset

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import classification_report, confusion_matrix

COLUMNS = ['buying', 'maintenance', 'doors', 'persons', 'luggage_boot', 'safety', 'class']
URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data'

def load_car_data():
    """Try a local car.data file first, then the UCI URL, then OpenML."""
    if os.path.exists('car.data'):
        print('Loaded from local file: car.data')
        return pd.read_csv('car.data', header=None, names=COLUMNS)
    try:
        df = pd.read_csv(URL, header=None, names=COLUMNS)
        print('Loaded from UCI repository')
        return df
    except Exception as e:
        print('UCI download failed:', e)
    try:
        from sklearn.datasets import fetch_openml
        raw = fetch_openml('car', version=2, as_frame=True).frame
        raw.columns = COLUMNS
        print('Loaded from OpenML')
        return raw
    except Exception as e:
        raise RuntimeError(
            'Could not load the data. Download car.data from '
            'https://archive.ics.uci.edu/ml/datasets/Car+Evaluation and place it '
            'in the same folder as this notebook, then re-run.') from e

data = load_car_data()
print(data.shape)
data.head()

Loaded from UCI repository
(1728, 7)


,buying,maintenance,doors,persons,luggage_boot,safety,class
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


### Step 2: Label encoding with `LabelEncoder()`

In [2]:
data = data.astype(str)
label_encoders = {}
for col in data.columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le

print('Target classes (encoded order):', list(label_encoders['class'].classes_))
data.head()

Target classes (encoded order): ['acc', 'good', 'unacc', 'vgood']


,buying,maintenance,doors,persons,luggage_boot,safety,class
0,3,3,0,0,2,1,2
1,3,3,0,0,2,2,2
2,3,3,0,0,2,0,2
3,3,3,0,0,1,1,2
4,3,3,0,0,1,2,2


### Step 3: Extract the label variable with `pop()`

In [3]:
label = data.pop('class')
print(data.columns.tolist())
label.value_counts()

['buying', 'maintenance', 'doors', 'persons', 'luggage_boot', 'safety']


class
2    1210
0     384
1      69
3      65
Name: count, dtype: int64

### Step 4: Split into training and testing data (10% test)

In [4]:
features_train, features_test, label_train, label_test = train_test_split(
    data, label, test_size=0.1, random_state=0)
print('Train:', features_train.shape, ' Test:', features_test.shape)

Train: (1555, 6)  Test: (173, 6)


### Step 5: Build the decision tree classifier

In [5]:
decision_tree = DecisionTreeClassifier(random_state=0)
decision_tree.fit(features_train, label_train)
label_predicted = decision_tree.predict(features_test)

### Step 6: Score on the test data

In [6]:
print('Decision tree test accuracy:', decision_tree.score(features_test, label_test))

Decision tree test accuracy: 0.9421965317919075


### Step 7: Classification report

In [7]:
class_names = list(label_encoders['class'].classes_)
print(classification_report(label_test, label_predicted))
print('Class mapping:', dict(enumerate(class_names)))

              precision    recall  f1-score   support

           0       0.88      0.88      0.88        42
           1       0.67      0.86      0.75         7
           2       0.98      0.97      0.98       120
           3       1.00      0.75      0.86         4

    accuracy                           0.94       173
   macro avg       0.88      0.87      0.87       173
weighted avg       0.95      0.94      0.94       173

Class mapping: {0: 'acc', 1: 'good', 2: 'unacc', 3: 'vgood'}


## Activity 4.02 – Random Forest Classification for Your Car Rental Company
### Step 1: Steps 1–4 of Activity 4.01
Already done above (data loaded, encoded, label popped, 90/10 split).

### Steps 2–3: Create and train a Random Forest

In [8]:
random_forest = RandomForestClassifier(random_state=0)
random_forest.fit(features_train, label_train)
rf_predicted = random_forest.predict(features_test)
print('Random forest test accuracy:', random_forest.score(features_test, label_test))

Random forest test accuracy: 0.9479768786127167


### Step 4: Confusion matrix

In [9]:
rf_cm = confusion_matrix(label_test, rf_predicted)
print(pd.DataFrame(rf_cm, index=class_names, columns=class_names))

       acc  good  unacc  vgood
acc     39     2      1      0
good     2     4      0      1
unacc    2     0    118      0
vgood    1     0      0      3


### Step 5: Classification report

In [10]:
print(classification_report(label_test, rf_predicted))

              precision    recall  f1-score   support

           0       0.89      0.93      0.91        42
           1       0.67      0.57      0.62         7
           2       0.99      0.98      0.99       120
           3       0.75      0.75      0.75         4

    accuracy                           0.95       173
   macro avg       0.82      0.81      0.81       173
weighted avg       0.95      0.95      0.95       173



### Step 6: Feature importance
(The book writes `.feature_importance_`; the actual scikit-learn attribute is `.feature_importances_`.)

In [11]:
print(random_forest.feature_importances_)
pd.Series(random_forest.feature_importances_, index=data.columns).sort_values(ascending=False)

[0.19354086 0.1506057  0.05862039 0.22793704 0.07902146 0.29027454]


safety          0.290275
persons         0.227937
buying          0.193541
maintenance     0.150606
luggage_boot    0.079021
doors           0.058620
dtype: float64

### Step 7: Repeat with Extremely Randomized Trees

In [12]:
extra_trees = ExtraTreesClassifier(random_state=0)
extra_trees.fit(features_train, label_train)
et_predicted = extra_trees.predict(features_test)

print('Extra trees test accuracy:', extra_trees.score(features_test, label_test))
print()
print('Confusion matrix:')
print(pd.DataFrame(confusion_matrix(label_test, et_predicted), index=class_names, columns=class_names))
print()
print(classification_report(label_test, et_predicted))
print('Feature importances:')
print(extra_trees.feature_importances_)

Extra trees test accuracy: 0.976878612716763

Confusion matrix:
       acc  good  unacc  vgood
acc     42     0      0      0
good     2     5      0      0
unacc    2     0    118      0
vgood    0     0      0      4

              precision    recall  f1-score   support

           0       0.91      1.00      0.95        42
           1       1.00      0.71      0.83         7
           2       1.00      0.98      0.99       120
           3       1.00      1.00      1.00         4

    accuracy                           0.98       173
   macro avg       0.98      0.92      0.94       173
weighted avg       0.98      0.98      0.98       173

Feature importances:
[0.19821279 0.17106337 0.04440978 0.23421321 0.07781975 0.2742811 ]


### Optional: Cross-validation (from the chapter text)

In [13]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(RandomForestClassifier(random_state=0), features_train, label_train,
                         cv=5, scoring='accuracy')
print(scores, '\nMean accuracy:', scores.mean())

[0.98713826 0.94855305 0.95819936 0.98713826 0.95498392] 
Mean accuracy: 0.9672025723472668


## Summary
- A single decision tree is accurate but prone to overfitting.
- Random forests and extra trees add randomness and majority voting, which reduces overfitting.
- Feature importances show which attributes (typically **safety** and **persons**) drive the classification most.
- Metrics (precision, recall, F1) are all derived from the confusion matrix.